### Data Reading

**Reading the json file**

In [0]:
df_json = spark.read.format("json")\
            .option('multiLine', True)\
                .load("/Volumes/workspace/default/data/drivers.json")

In [0]:
display(df_json)

In [0]:
# looking for the available files in particular location
dbutils.fs.ls('/Volumes/workspace/default/data')

CSV File Reader

In [0]:
df = spark.read.format("csv").option('inferSchema', False)\
            .option("header", True)\
                .load("/Volumes/workspace/default/data/BigMart Sales.csv")

In [0]:
display(df)

In [0]:
df.show()

🔑 Note: display is much more beautiful, so prefere display over show.

### Schema Definition

In [0]:
## seeing the schema
df.printSchema()

In [0]:
my_ddl_schema = '''
                    Item_Identifier STRING,
                    Item_Weight STRING,
                    Item_Fat_Content STRING, 
                    Item_Visibility DOUBLE,
                    Item_Type STRING,
                    Item_MRP DOUBLE,
                    Outlet_Identifier STRING,
                    Outlet_Establishment_Year INT,
                    Outlet_Size STRING,
                    Outlet_Location_Type STRING, 
                    Outlet_Type STRING,
                    Item_Outlet_Sales DOUBLE 

                ''' 

In [0]:
df = spark.read.format("csv")\
            .schema(my_ddl_schema)\
            .option("header", True)\
            .load("/Volumes/workspace/default/data/BigMart Sales.csv") 

In [0]:
display(df)

In [0]:
df.printSchema()

### StructType() Schema

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
my_struct_schema = StructType([ 
                StructField('Item_Identifier',StringType(),True), 
                StructField('Item_Weight',StringType(),True), 
                StructField('Item_Fat_Content',StringType(),True), 
                StructField('Item_Visibility',StringType(),True), 
                StructField('Item_MRP',StringType(),True), 
                StructField('Outlet_Identifier',StringType(),True), 
                StructField('Outlet_Establishment_Year',StringType(),True), 
                StructField('Outlet_Size',StringType(),True), 
                StructField('Outlet_Location_Type',StringType(),True), 
                StructField('Outlet_Type',StringType(),True), 
                StructField('Item_Outlet_Sales',StringType(),True)

])

In [0]:
df = spark.read.format('csv') \
        .schema(my_struct_schema) \
        .option("header", "true") \
        .load("/Volumes/workspace/default/data/BigMart Sales.csv")

In [0]:
df.printSchema()

### Select Transformation

Selecting the particular column, similar to SQL SELECT statement.

In [0]:
df.display()

In [0]:
df_selected = df.select('Item_Identifier', 'Item_Identifier', 'Item_Fat_Content')
df_selected.display()

In [0]:
## Alternative
df.select(col('Item_Identifier'), \
          col('Item_Identifier'), \
          col('Item_Fat_Content')).display()


> **ALIAS**

In [0]:
df.select(col('Item_Identifier').alias('Item_id')).display()

### **Filter/WHERE**

Helps in filtering the data based on certain condition/s.

Scenario - 1

get the data where `Item_fat_content == Regular`


In [0]:
df.filter(col('Item_Fat_Content') == 'Regular').display()

Scenario - 2

Fetch those record `Item_MRP == Dairy AND Item_weight < 10`.

In [0]:
df.filter(
    (col('Item_MRP') == 'Dairy') &
     (col('Item_Weight').cast("float") < 10)  # .cast("float") is required to convert the column to float
).display()

Scenario - 3

Fetch the data from Tier1 or Tier2 and outlet size = Null.

In [0]:
df.filter(
    (col('Outlet_Type').isin('Tier 1', 'Tier 2')) &
    (col('Outlet_Location_Type').isNull())
).display()

> **`withColumnRenamed`**

In [0]:
df.withColumnRenamed('Item_Weight', 'Item_Wt').display()

### **`withColumn`**

Used to modify the existing column.

In [0]:
# creating new column 
df = df.withColumn('flag', lit('new'))  # lit is used to assign constant value to a column
df.display()

In [0]:
df.withColumn('multiply',
              col('Item_Weight').cast('float') * col('Outlet_Size').cast('float') * 0.2
).display()

%md
### **`regexp_replace`**

regexp_replace is a powerful function in Spark SQL and PySpark used to search for a regular expression pattern in a string column and replace it with a new string.

It searches the target text globally by default, replacing all occurrences of the matching pattern

In [0]:
df.withColumn('Item_Fat_Content', regexp_replace(
    col('Item_Fat_Content'), 'Regular', 'reg'))\
        .withColumn('Item_Fat_Content', regexp_replace(
            col('Item_Fat_Content'), 'Low Fat','LF')).display()


### *Type Casting* with `cast`

In [0]:
df = df.withColumn('Item_Weight', col('Item_Weight').cast('float'))

In [0]:
df.printSchema()

### **Sort / OrderBy**

In [0]:
# Scenario - 1
# sort Item_weight in descending order

df.sort(col('Item_Weight').desc()).display()

In [0]:
# Scenario - 2
# Item_visibility in ascending order

df.sort(col('Item_Visibility').asc()).display()

In [0]:
# Scenario - 3
# sorting based on multiple column
df.sort(['Item_Weight', 'Item_Visibility'], 
        ascending=[False, True]).display()

### Limit

In [0]:
df.limit(10).display()

### Drop

In [0]:
df.drop('Item_Visibility').display()

In [0]:
# drop multiple columns
df.drop('Item_Visibility', 'Item_Weight').display()

### DROP_DUPLICATES

In [0]:
df.dropDuplicates().display()  # also known as 'd_dupe'

In [0]:
# droping duplicates row from a particular column
df.drop_duplicates(subset=['Item_Visibility']).display()

In [0]:
df.distinct().display()  # gives the distinct row same as drop_duplicstes

### Union and UnionByName

In [0]:
# preparing the DataFrame
data1 = [('1', 'Alex'),
         ('2', 'Bob')]

data2 = [('3', 'Charles'),
         ('4', 'Dia'),
         ('5', 'Eva')]

schema = 'id STRING, name STRING'

df1 = spark.createDataFrame(data1, schema)
df2 = spark.createDataFrame(data2, ['id', 'name'])

In [0]:
df1.display()

In [0]:
df2.display()

In [0]:
df1.union(df2).display()

In [0]:
data1 = [('Alex', '1'),
         ('Bob', '2')]

schema = 'name STRING, id STRING'

df1 = spark.createDataFrame(data1, schema)

df1.display()
df1.union(df2).display()

It's just goofed up that's why to overcome this we use `unionbyname`.

In [0]:
df1.unionByName(df2).display()

### String functions

* INITCAP()
* UPPER()
* LOWER()
* CONCAT()
* CONCAT_WS()
* SPLIT()
* SUBSTRING()
* REPLACE()
* TRIM()
* LTRIM()
* RTRIM()
* REGEXP_REPLACE()
* REGEXP_EXTRACT()
* SOUNDEX()

📖 **Task**: Read about all the available pyspark functions cuz we gonna use it a lot and perform a few here

In [0]:
# we will perform on Item_MRP column
# Initcap()
# Capitalize the Initial Letter of the words

df.select(
    initcap(col('Item_MRP')
)).display()


In [0]:
# Lower
df.select(
    lower(col('Item_MRP'))
).display()

In [0]:
# Upper
df.select(
    upper(col('Item_MRP'))
).display()

In [0]:
# Alias with Upper
df.select(
    upper(col('Item_MRP').alias('Upper_Item_MRP'))
).display()

### Date Functions

* Current_date()
* Date_add()
* Date_sub()
* Datediff()
* Months_between()
...

In [0]:
# Current_date()
df = df.withColumn('Curr_date', current_date())
display(df)

In [0]:
# Date_add
df = df.withColumn('Week_after', date_add('Curr_date', 7))

df.display()

In [0]:
# date_sub 
# date Subtract
df = df.withColumn('Week_before', date_sub('Curr_date', 7))

df.display()


In [0]:
# datediff
# Number of days between two dates
df = df.withColumn('days_between',
    datediff('Curr_date','Week_after')
)

df.display()

In [0]:
# months_between
# Number of months between two dates
df = df.withColumn('Months_between',
    months_between(col('Week_after'), col('Curr_date'))
)

df.display()

### Date_format()

In [0]:
df = df.withColumn('Week_before',
                   date_format('Week_before', 'dd-MM-yyyy'))

df.display()

### Handling nulls

In [0]:
# Dropping null
df.dropna().display()  # ['any', 'all']
             # default is 'any'

In [0]:
# dropping froma particular column
df.dropna(subset=['Outlet_Location_Type']).display()

In [0]:
# filling the null values
df.fillna('NotAvailable').display()

In [0]:
# filling for a particular column
df.fillna('NotAvilable', subset=['Outlet_Location_Type']).display()

### Split and Indexing

In [0]:
df.withColumn('Item_Outlet_sales', split(col('Item_Outlet_Sales'), ' ')).display()

In [0]:
# Indexing
# df.withColumn('Item_Outlet_sales', 
#               split(col('Item_Outlet_sales')[1], r' ')).display()


### Explode

In [0]:
df_exp = df.withColumn('Item_Outlet_sales', 
              split(col('Item_Outlet_Sales'), ' '))

df.display()

In [0]:
df_exp.printSchema()

In [0]:
df_exp.withColumn('Item_Outlet_sales', 
              explode(col('Item_Outlet_sales'))).display()

In [0]:
df_exp.withColumn('Type1_flag', array_contains(col('Item_Outlet_sales'), 'Type1')).display()

### GroupBy

In [0]:
# Scenario - 1
df_exp.groupBy('Item_MRP').agg(sum('Outlet_Identifier')).display()

In [0]:
# Average mrp
df_exp.groupBy('Item_MRP').agg(avg('Outlet_Identifier')).display()

In [0]:
# group by on two columns
df.groupBy('Item_MRP', 'Outlet_Location_Type').agg(sum('Outlet_Identifier')).display()

In [0]:
# group by on two columns with two aggregate function
df.groupBy('Item_MRP', 'Outlet_Location_Type').\
    agg(sum('Outlet_Identifier'),
        avg('Outlet_Identifier')).display()

# Spark Advanced Functions

### Collect_List

In PySpark, there are two completely different ways to "collect a list" depending on your goal: using `pyspark.sql.functions.collect_list()` to aggregate rows into a column of arrays, or using `DataFrame.collect()` to transfer entire rows back to your Python script.

In [0]:
data = [
        ('User1', 'book1'),
        ('User1', 'book2'),
        ('User2', 'book2'),
        ('User2', 'book'),
        ('User3', 'book1'),
]

schema = 'user string, book string'

df_book = spark.createDataFrame(data, schema)

df_book.display()

In [0]:
df_book.groupBy('user').\
    agg(collect_list('book')).display()

### Pivot

In [0]:
df.groupBy('Item_MRP').\
    pivot('Outlet_Location_Type').\
        agg(avg('Outlet_Identifier')).display()

### When-Otherwise

Pyspark `if-else`

In [0]:
df.withColumn('Veg_Flag',
    when(col('Item_MRP') == 'Meat', 'Non_Veg')\
    .otherwise('Veg')).display()


In [0]:
df.withColumn('Expensive_Flag',
              when(col('Outlet_Identifier').cast('float') > 100, 'Expensive')\
              .when(col('Outlet_Identifier').cast('float') > 50, 'Moderate')\
              .otherwise('Cheap')
).display()

### Joins

* Inner Join
* Left Join
* Right Join
* Outer Join
* ANTI JOIN

In [0]:
dataj1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)

In [0]:
df1.display()

In [0]:
df2.display()

In [0]:
# Inner join
df1.join(df2,df1.dept_id == df2.dept_id,'inner').display()

In [0]:
# Left join
df1.join(df2,df1.dept_id == df2.dept_id,'left').display()


In [0]:
# Right join
df1.join(df2,df1.dept_id == df2.dept_id,'right').display()

In [0]:
# Anti join
df1.join(df2,df1.dept_id == df2.dept_id,'anti').display()

### Window functions

Special Functions which perform `Row Level Manipulation` in place.

* Some Popular Window Functions are: 
    * Row_Number()
    * Rank()
    * DenseRank()
    ...

In [0]:
# Row_Number
# Unique Identifier for the records
from pyspark.sql.window import Window

### Window Functions in PySpark: Using `.over()`

In PySpark, `.over()` is used to apply a window function (like ranking, lead/lag, or running totals) to a DataFrame. It binds an aggregate or analytical function to a specific `WindowSpec` (which defines how data is grouped and ordered), allowing you to perform calculations across a subset of rows **without reducing the overall row count**.

---

#### Core Syntax

To use `.over()`, you generally follow three steps:

1. **Define the Window:** Use `Window.partitionBy()` (to group data) and `Window.orderBy()` (to sort within groups).

2. **Select the Function:** Choose from PySpark's functions library (e.g., `row_number()`, `rank()`, `sum()`, `avg()`, `lag()`).

3. **Apply `.over()`:** Use `.over(window)` within `.withColumn()`.

##### Code Example

```python
from pyspark.sql import Window
import pyspark.sql.functions as F

# 1. Define the window spec
windowSpec = Window.partitionBy("department").orderBy(F.desc("salary"))

# 2 & 3. Select the function and apply .over()
df_ranked = df.withColumn("rank", F.rank().over(windowSpec))

In [0]:
df.withColumn('rowCol', row_number().\
    over(Window.orderBy('Item_Identifier'))).display()

In [0]:
# Rank()
df.withColumn('rankCol', rank().\
    over(Window.orderBy('Item_Identifier'))).display()

In [0]:
# rank in descending order
# RDenseank()
df.withColumn(
    'descRankCol', 
    rank().over(Window.orderBy(col('Item_Identifier').desc())))\
        .withColumn(
        'denseRank', 
        dense_rank().over(Window.orderBy(col('Item_Identifier').desc()))).display()


In [0]:
## Cumulative Sum
df.withColumn('cumSum', sum('Outlet_Identifier').\
    over(Window.orderBy('Item_MRP'))).display()

In [0]:
df.withColumn('cumSum', sum('Outlet_Identifier').\
    over(Window.orderBy('Item_MRP').\
        rowsBetween(Window.unboundedPreceding, Window.currentRow))).display()

In [0]:
df.withColumn('totalSum', sum('Outlet_Identifier').\
    over(Window.orderBy('Item_MRP').\
        rowsBetween(Window.unboundedFollowing, Window.currentRow))).display()

### User Defined Function (udf)

Transformation which cannot be done with availbale function we shall create `udf`.

In [0]:
# Step 1
# define python function
def square(x):
    return x*x

# Step 2
# register python function as UDF
squareUDF = udf(square, IntegerType())

# Step 3
# apply UDF to column
df.withColumn('square', squareUDF(col('Outlet_Identifier').cast('float'))).display()

### Data writing with Pyspark

In [0]:
# CSV
# df.write.format('csv').\
#     save('dbfs:/FileStore/tables/sales.csv')

df.write.format('csv').\
    save(
    format='csv',
    #mode='overwrite',
    #header=True,
    path='/Volumes/workspace/default/data/Sales.csv'
)

In [0]:
# checking our created file 
dbutils.fs.ls('/Volumes/workspace/default/data')

### Available writing mode in PySpark

* Append
* Overwrite
* Error - when file is already there
* Ignore - no error when the same name is always present.

In [0]:
df.write.format('csv').\
    save(
    format='csv',
    mode='append',
    #header=True,
    path='/Volumes/workspace/default/data/Sales.csv')



In [0]:
df.write.format('csv').\
    save(
    format='csv',
    mode='error',
    #header=True,
    path='/Volumes/workspace/default/data/Sales.csv')



In [0]:
df.write.format('csv').\
    save(
    format='csv',
    mode='ignore',
    #header=True,
    path='/Volumes/workspace/default/data/Sales.csv')



### `Parquet` file format

In [0]:
df.write.format('parquet').\
    save(
    mode='overwrite',
    #header=True,
    path='/Volumes/workspace/default/data/Sales.csv')



## Parquet vs Delta Lake

## What is Parquet?

Parquet is an open-source columnar storage file format designed for efficient data storage and retrieval.

### Features
- Columnar storage
- High compression
- Fast analytical queries
- Schema support
- Widely supported by Spark, Pandas, Hive, Presto, Trino, etc.

### Limitations
- No ACID transactions
- No versioning or time travel
- No built-in data consistency guarantees
- Concurrent writes can cause issues

---

## What is Delta Lake?

Delta Lake is an open-source storage layer built on top of Parquet files that adds reliability and advanced data management features.

### Features
- ACID Transactions
- Time Travel
- Schema Enforcement
- Schema Evolution
- Data Versioning
- Audit History
- Concurrent Read/Write Support
- MERGE, UPDATE, DELETE Operations

### Components
- Parquet data files
- `_delta_log` transaction log



### Table


In [0]:
df.write.format('parquet').\
    mode('overwrite').\
    saveAsTable('my_table')



In [0]:
df.write.format('delta').\
    mode('overwrite').\
    saveAsTable('my_table')



### Managed vs External Tables

# Managed vs External Tables

## Managed Table

A **Managed Table** is a table where Spark/Hive/Databricks manages both the table metadata and the underlying data files.

### Characteristics

- Data is stored in the default warehouse location.
- Metadata is stored in the metastore.
- Spark/Hive owns the data.
- Dropping the table removes both metadata and data files.

### Example

```sql
CREATE TABLE employees (
    id INT,
    name STRING
);
```

### Storage Location

```text
warehouse/
└── employees/
    ├── part-00001.parquet
    └── part-00002.parquet
```

### Drop Table

```sql
DROP TABLE employees;
```

Result:

- Metadata deleted ✅
- Data files deleted ✅

---

## External Table

An **External Table** is a table where Spark/Hive manages only the metadata, while the data remains at a user-specified location.

### Characteristics

- Data stored at a custom location.
- Metadata stored in metastore.
- User owns the data.
- Dropping the table removes only metadata.
- Physical data files remain unchanged.

### Example

```sql
CREATE EXTERNAL TABLE employees_ext (
    id INT,
    name STRING
)
LOCATION '/mnt/data/employees';
```

### Storage Location

```text
/mnt/data/employees/
├── part-00001.parquet
└── part-00002.parquet
```

### Drop Table

```sql
DROP TABLE employees_ext;
```

Result:

- Metadata deleted ✅
- Data files remain ✅

---

## Comparison

| Feature | Managed Table | External Table |
|----------|--------------|---------------|
| Data Location | Default Warehouse | User-Specified Location |
| Metadata | Metastore | Metastore |
| Data Ownership | Spark/Hive | User |
| DROP TABLE | Deletes Data + Metadata | Deletes Metadata Only |
| Data Sharing | Limited | Easy |
| Portability | Lower | Higher |
| Data Lake Usage | Less Common | More Common |

---

## Databricks Delta Examples

### Managed Delta Table

```sql
CREATE TABLE sales
USING DELTA
AS
SELECT * FROM sales_df;
```

Default Location:

```text
dbfs:/user/hive/warehouse/sales/
```

---

### External Delta Table

```sql
CREATE TABLE sales_ext
USING DELTA
LOCATION 'dbfs:/mnt/datalake/sales';
```

Custom Location:

```text
dbfs:/mnt/datalake/sales/
```

---

## How to Check Table Type

```sql
DESCRIBE EXTENDED employees;
```

Output:

```text
Type: MANAGED
```

or

```text
Type: EXTERNAL
```

---

## Interview Answer

### Managed Table
Spark/Hive manages both metadata and data. When the table is dropped, both metadata and underlying data files are deleted.

### External Table
Spark/Hive manages only metadata. Data is stored at a user-defined location and remains intact even if the table is dropped.

### Rule of Thumb

- Use **Managed Tables** for temporary or fully controlled datasets.
- Use **External Tables** for production data lakes, shared storage, and Delta Lake implementations.

# Spark SQL

In [0]:
# Creating temproary view
df.createOrReplaceTempView("my_temp_view")

In [0]:
%sql
SELECT * from my_temp_view 
WHERE Item_Fat_Content = 'Low Fat';

In [0]:
# Saving sql result into dataFrame
df_sql = spark.sql("SELECT * from my_temp_view \
    WHERE Item_Fat_Content = 'Low Fat';")


In [0]:
df_sql.display()